# Null Handling, Missing Data Strategies, and Quality Validation

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 07_nulls_and_data_quality.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark handles null and NaN values, using na.fill(), na.drop(), and na.replace(), null-safe equality (<=>), and building validation pipelines.
- **Why does this topic exist?**: Real-world data is inherently dirty. Missing keys break joins, skew partitions, and corrupt aggregations.
- **What problem does it solve?**: Silent null propagation causes misleading report totals or runtime NullPointerExceptions.
- **Where is it used in real-world Data Engineering?**: Auditing data quality gates in production Lakehouse architectures.

---

## 2. Null Semantics in Spark SQL

```
+-------------------------------------------------------------------------------+
| Spark SQL Null Handling Rules                                                 |
+-------------------------------------------------------------------------------+
| Standard Equality (col == null)     | Always returns NULL (unknown)           |
| Null-Safe Equality (col <=> null)   | Returns TRUE if both null, FALSE if one |
| Standard Aggregations (SUM, AVG)    | Silently ignores NULL values            |
| COUNT(*)                            | Counts ALL rows including nulls         |
| COUNT(col)                          | Counts only NON-NULL rows               |
+-------------------------------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Quality_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Auditing Missing Data in `/opt/spark-data/quality/orders_dirty.csv`

In [ ]:
dirty_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/quality/orders_dirty.csv")

print(f"Total rows in dirty dataset: {dirty_df.count()}")
dirty_df.printSchema()

# Profile null counts across all columns
null_counts = dirty_df.select([
    F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), c)).alias(c)
    for c in dirty_df.columns
])

print("Null counts per column:")
null_counts.show()

## 4. Cleaning Strategies: `na.fill()`, `na.drop()`, and `coalesce()`

In [ ]:
# Strategy 1: Drop rows where critical primary keys are null
cleaned_step1 = dirty_df.na.drop(subset=["order_id", "customer_id"])

# Strategy 2: Fill null discounts with 0.0 and null status with 'Pending'
cleaned_step2 = cleaned_step1.na.fill({
    "discount": 0.0,
    "status": "Pending",
    "channel": "Unknown"
})

print(f"Rows before: {dirty_df.count()} -> Rows after key drop: {cleaned_step1.count()}")
cleaned_step2.filter("discount IS NULL OR status IS NULL").show()
print("Cleaned sample:")
cleaned_step2.show(5)

---

## Experiment: Null-Safe Equality (<=>) in Filter and Joins

- **1. What are we testing?**: Demonstrate that standard'=='evaluates to null when comparing null values, whereas'<=>'evaluates deterministically.
- **2. What do we expect?**: df.filter(col('a') == col('b')) drops rows where both are null; col('a').eqNullSafe(col('b')) retains them.

In [ ]:
test_nulls = spark.createDataFrame([
    (1, "A", "A"),
    (2, None, None),
    (3, "B", None)
], ["id", "val1", "val2"])

print("Standard Equality (val1 == val2):")
test_nulls.filter(F.col("val1") == F.col("val2")).show()

print("Null-Safe Equality (val1.eqNullSafe(val2)):")
test_nulls.filter(F.col("val1").eqNullSafe(F.col("val2"))).show()

**3. What happened?**: Standard equality dropped row 2 (both None); null-safe equality correctly preserved row 2.  
**4. Why did it happen?**: In ANSI SQL three-valued logic, NULL == NULL produces NULL (falsy in filters). eqNullSafe explicitly tests for equivalence.  
**5. Practical Engineering Lesson**: Always use eqNullSafe() when joining or filtering nullable keys where null-to-null matching is required.

---

## Practical Exercises

### Exercise 1 (Beginner): Filter Non-Null Valid Records
**Problem Statement**: Filter dirty_df to retain only rows where both quantity and unit_price are NOT null and greater than zero.  
**Input / Context**: dirty_df

In [ ]:
# Exercise 1: Valid numeric records

### Exercise 2 (Intermediate): Replace Invalid String Values with na.replace()
**Problem Statement**: Use na.replace() to standardize channel values: map'App'to'Mobile'and'N/A'to'Unknown'.  
**Input / Context**: dirty_df channel column

In [ ]:
# Exercise 2: na.replace mapping

### Exercise 3 (Advanced): Data Quality Metric Report Card
**Problem Statement**: Calculate the percentage completeness (non-null %) for every column in dirty_df and display as a summary DataFrame.  
**Input / Context**: dirty_df

In [ ]:
# Exercise 3: Completeness report

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Filter Non-Null Valid Records):
```python
valid = dirty_df.filter(F.col("quantity").isNotNull() & (F.col("quantity") > 0) & F.col("unit_price").isNotNull() & (F.col("unit_price") > 0))
print("Valid count:", valid.count())
```

#### Solution 2 (Replace Invalid String Values with na.replace()):
```python
standardized = dirty_df.na.replace({"App":"Mobile","N/A":"Unknown"}, subset=["channel"])
standardized.groupBy("channel").count().show()
```

#### Solution 3 (Data Quality Metric Report Card):
```python
total_rows = dirty_df.count()
completeness_exprs = [
    F.round((F.count(c) / total_rows) * 100, 2).alias(f"{c}_pct_complete")
    for c in dirty_df.columns
]
dirty_df.select(completeness_exprs).show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Null Semantics**: NULL == NULL produces NULL; use eqNullSafe() for null-aware matching.
- **na Package**: na.fill(), na.drop(), na.replace() provide high-level missing data management.
- **Aggregation Impact**: Standard aggregations ignore nulls, but COUNT(*) includes them.

### Essential Technical Terms
- `eqNullSafe`: Null-safe equality operator (<=>).
- `na.fill`: Replaces null values with specified constants.
- `na.drop`: Drops rows containing null values in designated subsets.

### Common Engineering Mistakes
- **Using == to check for null**: df.filter(col('x') == None) fails silently; must use col('x').isNull().
- **Forgetting that Joins Drop Null Keys**: Standard equi-joins drop rows with null keys on both sides.

### Interview Questions on This Topic
- **Q: What does COUNT(col) return if all values in that column are null?**
  - *A*: It returns 0.
- **Q: How does eqNullSafe differ from standard equality?**
  - *A*: It returns True when comparing NULL to NULL.

### What to Study Next
Open 03_dataframe_and_spark_sql/08_udfs_and_pandas_udfs.ipynb to compare standard UDFs with Arrow Pandas UDFs.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')